# Analisis Sentimen MBG dari X Posts Menggunakan Support Vector Machine (SVM)

Notebook ini menganalisis opini publik tentang **MBG** dari data post X/Twitter pada file `mbg_x_posts.csv`. Tujuannya adalah mengelompokkan post menjadi tiga sentimen: **positif**, **netral**, dan **negatif**.

Bayangkan notebook ini seperti alur kerja riset kecil: pertama kita membuka data, lalu membersihkan teks yang berantakan, memberi label sentimen, melatih model komputer agar mengenali pola kata, kemudian melihat hasilnya. Model yang dipakai adalah **Support Vector Machine (SVM)**, yaitu metode machine learning yang sering dipakai untuk klasifikasi teks.

Hal penting yang perlu dipahami:
- Jika dataset sudah punya kolom `sentiment`, notebook akan memakai label tersebut sebagai jawaban benar.
- Jika dataset belum punya kolom `sentiment`, notebook membuat label awal memakai daftar kata positif dan negatif. Ini disebut **weak label**, artinya label dibuat otomatis sebagai baseline, bukan hasil penilaian manusia.
- Karena SVM membutuhkan contoh berlabel, kualitas hasil model sangat bergantung pada kualitas label yang dipakai.

Output utama notebook ini adalah:
1. Akurasi model dalam persen dan jumlah prediksi benar pada data test.
2. Jumlah dan persentase post yang diprediksi sebagai positif, netral, dan negatif.
3. Data post untuk setiap sentimen, termasuk file CSV terpisah agar mudah dibuka kembali.


## 1. Instalasi dan Import Library

Bagian ini menyiapkan alat kerja yang dibutuhkan notebook. Library seperti `pandas` dipakai untuk membaca tabel, `scikit-learn` untuk membuat model SVM, dan `matplotlib`/`seaborn` untuk membuat grafik. Jika notebook dijalankan di Google Colab dan ada library yang belum tersedia, notebook akan mencoba memasangnya otomatis.


### Penjelasan Cell Kode 1: Mengecek Library

Cell ini mengecek apakah library yang dibutuhkan sudah tersedia. Jika ada yang belum ada, notebook mencoba menginstalnya. Untuk orang awam, library bisa dibayangkan seperti kotak alat: sebelum bekerja, kita memastikan semua alat penting sudah ada.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "joblib": "joblib",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
}

missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already installed.")


### Penjelasan Cell Kode 2: Menyiapkan Alat dan Lokasi Data

Cell ini mengimpor alat yang akan dipakai di seluruh notebook, mengatur gaya grafik, menentukan angka acak tetap agar hasil eksperimen bisa diulang, lalu mencari file dataset. Jika dijalankan di Colab, notebook mencoba memakai Google Drive; jika dijalankan lokal, notebook memakai file `mbg_x_posts.csv` di folder yang sama.


In [ ]:
import html
import re
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
DATASET_NAME = "mbg_x_posts.csv"
TOPIC_NAME = "MBG"

# Mount Google Drive jika notebook dijalankan di Google Colab.
try:
    from google.colab import drive

    drive.mount("/content/drive")
    print("Google Drive mounted.")
except ModuleNotFoundError:
    print("Not running in Google Colab. Using local file fallback.")

# Ubah path ini jika file berada di folder Google Drive lain.
DRIVE_DATA_PATH = Path(f"/content/drive/MyDrive/{DATASET_NAME}")
LOCAL_DATA_PATH = Path(DATASET_NAME)

if DRIVE_DATA_PATH.exists():
    DATA_PATH = DRIVE_DATA_PATH
elif LOCAL_DATA_PATH.exists():
    DATA_PATH = LOCAL_DATA_PATH
else:
    raise FileNotFoundError(
        f"Dataset tidak ditemukan. Simpan {DATASET_NAME} di MyDrive atau ubah DRIVE_DATA_PATH."
    )

print(f"Using dataset: {DATA_PATH}")


## 2. Teori Singkat Support Vector Machine (SVM)

**Support Vector Machine (SVM)** adalah algoritma supervised learning yang mencari batas pemisah terbaik antar kelas. Untuk klasifikasi, SVM membangun sebuah **hyperplane** yang memisahkan data dengan margin sebesar mungkin.

Konsep utama SVM:
- **Hyperplane**: garis/bidang keputusan yang memisahkan kelas. Pada data teks dengan banyak fitur TF-IDF, hyperplane berada pada ruang berdimensi tinggi.
- **Margin**: jarak antara hyperplane dan titik data terdekat dari setiap kelas. SVM memilih hyperplane dengan margin terbesar agar generalisasi lebih baik.
- **Support vectors**: data terdekat dengan hyperplane yang paling memengaruhi posisi batas keputusan.
- **Parameter C**: mengatur kompromi antara margin lebar dan kesalahan klasifikasi. Nilai C besar lebih ketat terhadap kesalahan, sedangkan C kecil memberi margin lebih longgar.

Untuk teks, SVM sering dipadukan dengan **TF-IDF** karena teks harus diubah menjadi representasi angka. Notebook ini memakai **LinearSVC**, versi SVM linear yang efisien untuk data teks berdimensi tinggi.


## 3. Load Dataset CSV

Bagian ini membaca file `mbg_x_posts.csv` menjadi tabel data. Teks dari media sosial kadang mengandung karakter aneh atau rusak, jadi fungsi pembaca CSV dibuat lebih aman agar proses tidak langsung gagal ketika menemukan karakter bermasalah.


### Penjelasan Cell Kode 3: Membaca File CSV

Cell ini membaca data post dari CSV. Fungsi `read_csv_safely` dibuat supaya karakter rusak di teks tidak membuat notebook berhenti. Output `Shape` menunjukkan jumlah baris dan kolom, daftar `Columns` menunjukkan nama kolom yang tersedia, dan `head()` menampilkan beberapa baris pertama sebagai contoh.


In [ ]:
def read_csv_safely(path: Path) -> pd.DataFrame:
    na_values = ["NULL", "null", "None", "none", ""]
    try:
        return pd.read_csv(
            path,
            encoding="utf-8",
            encoding_errors="replace",
            na_values=na_values,
            keep_default_na=True,
        )
    except TypeError:
        # Fallback untuk versi pandas lama yang belum punya parameter encoding_errors.
        with path.open("r", encoding="utf-8", errors="replace", newline="") as f:
            return pd.read_csv(f, na_values=na_values, keep_default_na=True)


df_raw = read_csv_safely(DATA_PATH)

print(f"Shape: {df_raw.shape}")
print("Columns:")
for column in df_raw.columns:
    print(f"- {column}")

df_raw.head()


## 4. Validasi Kolom dan Rapikan Metadata

Bagian ini memastikan data punya kolom minimal yang dibutuhkan, terutama `id` dan `text`. Setelah itu beberapa kolom angka, tanggal, dan bahasa dirapikan supaya mudah dianalisis. Ini seperti mengecek dan merapikan bahan mentah sebelum mulai memasak.


### Penjelasan Cell Kode 4: Mengecek dan Merapikan Data Dasar

Cell ini memastikan kolom `id` dan `text` ada, lalu merapikan tipe data. Kolom angka seperti like dan impression diubah menjadi angka, kolom tanggal diubah menjadi format tanggal, dan bahasa dirapikan menjadi huruf kecil. Di akhir, notebook menampilkan jumlah data, jumlah ID duplikat, dan jumlah teks kosong.


In [ ]:
required_columns = ["id", "text"]
missing_columns = [col for col in required_columns if col not in df_raw.columns]

if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

df = df_raw.copy()
df["id"] = df["id"].astype(str)
df["text"] = df["text"].fillna("").astype(str)

numeric_columns = [
    "like_count",
    "retweet_count",
    "reply_count",
    "quote_count",
    "impression_count",
    "bookmark_count",
    "is_retweet",
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0).astype(int)

for column in ["created_at", "created_at_date", "x_fetched_at"]:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

if "lang" in df.columns:
    df["lang"] = df["lang"].fillna("unknown").astype(str).str.lower().str.strip()

print(f"Total rows: {len(df):,}")
print(f"Duplicate post IDs: {df['id'].duplicated().sum():,}")
print(f"Empty text rows: {(df['text'].str.strip() == '').sum():,}")

df.head()


### Penjelasan Cell Kode 5: Ringkasan Dataset

Cell ini membuat tabel kecil berisi gambaran umum dataset: total post, jumlah akun unik, rentang tanggal, total likes, dan total impressions. Ringkasan ini membantu memahami ukuran dan cakupan data sebelum masuk ke model.


In [ ]:
summary_items = {
    "total_posts": len(df),
    "unique_authors": df["author_username"].nunique() if "author_username" in df.columns else np.nan,
    "date_min": df["created_at_date"].min() if "created_at_date" in df.columns else pd.NaT,
    "date_max": df["created_at_date"].max() if "created_at_date" in df.columns else pd.NaT,
    "total_likes": df["like_count"].sum() if "like_count" in df.columns else np.nan,
    "total_impressions": df["impression_count"].sum() if "impression_count" in df.columns else np.nan,
}

pd.DataFrame.from_dict(summary_items, orient="index", columns=["value"])


### Penjelasan Cell Kode 6: Melihat Bahasa dan Aktivitas Harian

Cell ini menampilkan jumlah post berdasarkan bahasa, lalu membuat grafik jumlah post per hari jika kolom tanggal tersedia. Grafik ini berguna untuk melihat kapan pembicaraan tentang MBG ramai atau sepi.


In [ ]:
if "lang" in df.columns:
    display(df["lang"].value_counts().rename_axis("lang").reset_index(name="count").head(15))

if "created_at_date" in df.columns:
    posts_per_day = df.dropna(subset=["created_at_date"]).set_index("created_at_date").resample("D").size()
    plt.figure(figsize=(12, 4))
    posts_per_day.plot()
    plt.title(f"Jumlah Post {TOPIC_NAME} per Hari")
    plt.xlabel("Tanggal")
    plt.ylabel("Jumlah Post")
    plt.tight_layout()
    plt.show()


## 5. Filter Data untuk Analisis Sentimen

Tidak semua post cocok langsung dipakai untuk analisis sentimen Bahasa Indonesia. Bagian ini menyaring data agar fokus pada post berbahasa Indonesia, menghapus retweet jika diinginkan, membuang teks kosong, dan menghindari post duplikat.


### Penjelasan Cell Kode 7: Menyaring Post yang Dipakai

Cell ini memilih data yang akan dianalisis sentimennya. Secara default notebook hanya mengambil post berbahasa Indonesia, membuang retweet, menghapus teks kosong, dan menghapus ID duplikat. Output membandingkan jumlah data sebelum dan sesudah filter.


In [ ]:
USE_INDONESIAN_ONLY = True
EXCLUDE_RETWEETS = True

filtered_df = df.copy()

if USE_INDONESIAN_ONLY and "lang" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["lang"].eq("in")].copy()

if EXCLUDE_RETWEETS and "is_retweet" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["is_retweet"].eq(0)].copy()

filtered_df = filtered_df[filtered_df["text"].str.strip().ne("")].drop_duplicates(subset="id").copy()

print(f"Rows before filter: {len(df):,}")
print(f"Rows after filter : {len(filtered_df):,}")
filtered_df[[col for col in ["id", "text", "author_username", "created_at_date", "lang", "is_retweet"] if col in filtered_df.columns]].head()


## 6. Preprocessing Teks X/Twitter

Teks media sosial biasanya berisi URL, mention, hashtag, simbol, dan kata-kata yang tidak terlalu membantu model. Preprocessing membersihkan hal-hal tersebut agar model lebih fokus pada kata yang membawa makna sentimen. Kata negasi seperti `tidak`, `bukan`, `gak`, dan `belum` tetap dipertahankan karena sangat penting untuk membedakan kalimat positif dan negatif.


### Penjelasan Cell Kode 8: Membersihkan Teks

Cell ini membuat daftar stopwords dan fungsi `clean_x_text`. Fungsi tersebut menghapus URL, mention, simbol, dan kata umum yang kurang berguna, lalu mengubah teks menjadi kata-kata penting. Hasilnya disimpan di kolom `clean_text`, yaitu versi teks yang lebih siap dipakai model.


In [ ]:
INDONESIAN_STOPWORDS = {
    "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk", "dengan", "atau",
    "pada", "adalah", "dalam", "karena", "sebagai", "akan", "sudah", "jadi",
    "ada", "aja", "saja", "juga", "lebih", "masih", "lagi", "kok", "nih",
    "dong", "lah", "pun", "nya", "rt", "https", "http", "co", "t", "x", "com",
    "the", "a", "an", "of", "to", "for", "in", "on", "at", "by", "is", "are",
}


def clean_x_text(text: str) -> str:
    text = html.unescape(str(text))
    text = text.replace("�", " ")
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r" \1 ", text)
    text = re.sub(r"[^0-9A-Za-zÀ-ÖØ-öø-ÿ_\s-]", " ", text)
    text = text.lower()
    tokens = re.findall(r"[a-zA-ZÀ-ÖØ-öø-ÿ_]+", text)
    tokens = [token for token in tokens if len(token) > 2 and token not in INDONESIAN_STOPWORDS]
    return " ".join(tokens)


filtered_df["clean_text"] = filtered_df["text"].apply(clean_x_text)
filtered_df[["text", "clean_text"]].head(10)


### Penjelasan Cell Kode 9: Membuang Teks yang Kosong Setelah Dibersihkan

Kadang sebuah post berisi hampir hanya URL, mention, atau simbol. Setelah dibersihkan, post seperti itu bisa menjadi kosong. Cell ini menghitung berapa yang kosong dan hanya menyimpan post yang masih punya kata bermakna untuk proses labeling dan modeling.


In [ ]:
empty_clean = filtered_df["clean_text"].str.len().eq(0).sum()
print(f"Jumlah teks kosong setelah cleaning: {empty_clean:,}")

df_text = filtered_df[filtered_df["clean_text"].str.len().gt(0)].copy()
print(f"Data siap labeling/model: {len(df_text):,} posts")


## 7. Label Sentimen

Model SVM belajar dari contoh yang sudah memiliki label. Jika data punya label manual, label itu digunakan. Jika belum ada, notebook membuat label otomatis memakai daftar kata dan frasa positif/negatif. Cara otomatis ini membantu membuat baseline cepat, tetapi hasil final tetap lebih kuat jika labelnya dicek atau dibuat oleh manusia.


### Penjelasan Cell Kode 10: Membuat atau Memakai Label Sentimen

Cell ini menentukan label sentimen untuk setiap post. Jika kolom `sentiment` sudah ada, label itu dipakai. Jika belum ada, notebook memakai daftar kata positif dan negatif untuk memberi label otomatis. Output tabel menunjukkan jumlah dan persentase label awal yang tersedia untuk model belajar.


In [ ]:
POSITIVE_WORDS = {
    "bagus", "baik", "mantap", "keren", "hebat", "top", "sukses", "berhasil",
    "dukung", "dukungannya", "apresiasi", "bangga", "percaya", "optimis", "semangat",
    "solid", "maju", "terbaik", "rapi", "tepat", "positif", "senang", "puas",
    "setuju", "layak", "lancar", "profesional", "transparan", "bermanfaat",
    "manfaat", "membantu", "terbantu", "sehat", "menyehatkan", "gratisan",
    "berguna", "peduli", "solutif", "efektif", "aman", "bersih", "enak",
    "lezat", "cukup", "terpenuhi", "merata", "tepat_sasaran",
}

NEGATIVE_WORDS = {
    "buruk", "jelek", "gagal", "kacau", "kecewa", "malu", "marah", "korup",
    "korupsi", "bobrok", "rusak", "parah", "aneh", "ribet", "batal", "bohong",
    "hoax", "masalah", "blunder", "protes", "kritik", "tidak", "bukan", "gak", "ga",
    "nggak", "belum", "jangan", "lambat", "curang", "salah", "ngaco", "memalukan",
    "tolol", "bodoh", "payah", "hancur", "krisis", "kontroversi", "keracunan",
    "racun", "beracun", "basi", "busuk", "muntah", "diare", "sakit", "jorok",
    "kotor", "bau", "mahal", "boros", "bocor", "utang", "pungli", "fiktif",
    "dipaksa", "dipolitisasi", "stop", "tolak", "mubazir", "sia", "zonk",
}

POSITIVE_PHRASES = {
    "program bagus", "program baik", "sangat membantu", "banyak membantu",
    "anak senang", "anak anak senang", "gizi terpenuhi", "nutrisi terpenuhi",
    "tepat sasaran", "berjalan lancar", "menu sehat", "makanan sehat",
    "makanan enak", "makan enak", "terima kasih", "patut diapresiasi",
    "layak didukung", "semoga lancar", "program bermanfaat",
}

NEGATIVE_PHRASES = {
    "tidak layak", "gak layak", "ga layak", "tidak jelas", "gak jelas", "ga jelas",
    "bikin kecewa", "sangat kecewa", "keracunan massal", "makanan basi", "makan basi",
    "menu basi", "makanan tidak layak", "program gagal", "stop mbg", "tolak mbg",
    "anggaran bocor", "rawan korupsi", "gizi buruk", "menu tidak layak",
    "anak keracunan", "siswa keracunan", "mubazir anggaran",
}

LABEL_MAP = {
    "netral": "neutral",
    "neutral": "neutral",
    "positif": "positive",
    "positive": "positive",
    "negatif": "negative",
    "negative": "negative",
}


def lexicon_sentiment_label(clean_text: str) -> str:
    tokens = clean_text.split()
    token_set = set(tokens)
    padded_text = f" {clean_text} "

    score = 0
    score += sum(1 for word in POSITIVE_WORDS if word in token_set)
    score -= sum(1 for word in NEGATIVE_WORDS if word in token_set)
    score += 2 * sum(1 for phrase in POSITIVE_PHRASES if f" {phrase} " in padded_text)
    score -= 2 * sum(1 for phrase in NEGATIVE_PHRASES if f" {phrase} " in padded_text)

    if score > 0:
        return "positive"
    if score < 0:
        return "negative"
    return "neutral"


df_model = df_text.copy()

if "sentiment" in df_model.columns:
    df_model["sentiment"] = (
        df_model["sentiment"]
        .fillna("neutral")
        .astype(str)
        .str.lower()
        .str.strip()
        .map(lambda value: LABEL_MAP.get(value, value))
    )
    LABEL_SOURCE = "existing sentiment column"
else:
    df_model["sentiment"] = df_model["clean_text"].apply(lexicon_sentiment_label)
    LABEL_SOURCE = "weak MBG lexicon labels"

label_counts = df_model["sentiment"].value_counts()
label_percent = (label_counts / label_counts.sum() * 100).round(2)

print(f"Label source: {LABEL_SOURCE}")
pd.DataFrame({"count": label_counts, "percent": label_percent})


### Penjelasan Cell Kode 11: Grafik Distribusi Label

Cell ini membuat grafik batang untuk memperlihatkan jumlah post pada setiap label sentimen. Grafik ini membantu melihat apakah datanya seimbang atau didominasi salah satu sentimen.


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df_model, x="sentiment", order=label_counts.index, palette="Set2")
plt.title(f"Distribusi Label Sentimen {TOPIC_NAME}")
plt.xlabel("Sentimen")
plt.ylabel("Jumlah Post")
plt.tight_layout()
plt.show()


### Penjelasan Cell Kode 12: Contoh Data Berlabel

Cell ini menampilkan beberapa contoh post secara acak beserta teks bersih dan labelnya. Tujuannya agar kita bisa mengecek secara manual apakah hasil cleaning dan label terlihat masuk akal.


In [ ]:
df_model[["text", "clean_text", "sentiment"]].sample(
    min(10, len(df_model)),
    random_state=RANDOM_STATE,
)


## 8. Split Data Train dan Test

Data dibagi menjadi dua bagian. **Data train** dipakai model untuk belajar, sedangkan **data test** disimpan untuk menguji apakah model bisa mengenali post yang belum pernah dilihat saat latihan. Pembagian dibuat seimbang supaya proporsi sentimen di train dan test tetap mirip.


### Penjelasan Cell Kode 13: Membagi Data Latihan dan Uji

Cell ini memisahkan kolom teks bersih sebagai input (`X`) dan sentimen sebagai target (`y`). Data lalu dibagi menjadi train dan test. Output menunjukkan jumlah data di masing-masing bagian dan komposisi sentimennya.


In [ ]:
X = df_model["clean_text"]
y = df_model["sentiment"]

if y.nunique() < 2:
    raise ValueError("Model SVM membutuhkan minimal dua kelas sentimen.")

stratify_target = y if y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_target,
)

print(f"Train size: {len(X_train):,}")
print(f"Test size : {len(X_test):,}")
print("\nDistribusi train:")
print(y_train.value_counts(normalize=True).round(3))
print("\nDistribusi test:")
print(y_test.value_counts(normalize=True).round(3))


## 9. Modeling dengan TF-IDF + Linear SVM

Komputer tidak memahami teks seperti manusia, jadi teks harus diubah menjadi angka. `TF-IDF` memberi bobot pada kata yang penting, lalu `LinearSVC` belajar memisahkan pola kata yang cenderung positif, netral, atau negatif. Grid search dipakai untuk mencoba beberapa kombinasi parameter dan memilih yang paling baik.


### Penjelasan Cell Kode 14: Melatih Model SVM

Cell ini membuat pipeline yang mengubah teks menjadi angka dengan TF-IDF lalu melatih SVM. Grid search mencoba beberapa pengaturan model dan memilih kombinasi terbaik berdasarkan macro F1, yaitu metrik yang memperhatikan semua kelas sentimen.


In [ ]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        min_df=2,
        max_df=0.9,
        sublinear_tf=True,
    )),
    ("svm", LinearSVC(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        dual=True,
        max_iter=5000,
    )),
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "svm__C": [0.1, 1, 5],
}

min_train_class_count = y_train.value_counts().min()
n_splits = min(5, min_train_class_count)

if n_splits >= 2:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    grid_search = GridSearchCV(
        estimator=svm_pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=1,
    )
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    print(f"CV folds: {n_splits}")
    print(f"Best CV macro F1: {grid_search.best_score_:.4f}")
    print(f"Best params: {grid_search.best_params_}")
else:
    best_model = svm_pipeline.fit(X_train, y_train)
    grid_search = None
    print("GridSearchCV dilewati karena ada kelas dengan jumlah data train kurang dari 2.")


## 10. Evaluasi Model

Setelah model selesai belajar, bagian ini mengecek performanya pada data test. Angka seperti accuracy, precision, recall, dan F1-score membantu melihat seberapa baik model menebak label. Jika label berasal dari weak label, metrik ini berarti model berhasil meniru aturan label otomatis, bukan berarti pasti sama dengan penilaian manusia.


### Penjelasan Cell Kode 15: Mengukur Performa Model

Cell ini meminta model menebak sentimen data test, lalu membandingkannya dengan label yang dianggap benar. `Accuracy` adalah persentase prediksi yang sama dengan label. `Classification report` memberi detail performa untuk setiap sentimen.


In [ ]:
y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Label source: {LABEL_SOURCE}")
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))


### Penjelasan Cell Kode 16: Confusion Matrix

Cell ini membuat confusion matrix, yaitu tabel visual yang menunjukkan label asli di sisi kiri dan prediksi model di sisi bawah. Angka diagonal berarti prediksi benar, sedangkan angka di luar diagonal menunjukkan kesalahan prediksi.


In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
)
plt.title(f"Confusion Matrix - SVM Sentiment {TOPIC_NAME}")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()


## 10.1 Hasil Model: Persentase, Jumlah, dan Data Post per Sentimen

Bagian ini menjawab pertanyaan utama tentang hasil model. Pertama, notebook menampilkan akurasi dalam bentuk persen dan jumlah prediksi benar pada data test. Setelah itu, model dipakai untuk memprediksi semua post yang lolos filter, lalu dihitung berapa post yang masuk sentimen positif, netral, dan negatif.

Selain ringkasan angka, bagian ini juga menyiapkan data post untuk setiap sentimen. Notebook menampilkan preview di layar dan menyimpan data lengkap ke file CSV terpisah, sehingga hasilnya mudah dicek kembali.


In [ ]:
INDONESIAN_LABELS = {
    "positive": "positif",
    "neutral": "netral",
    "negative": "negatif",
}
SENTIMENT_ORDER = ["positive", "neutral", "negative"]

correct_predictions = int((y_test == y_pred).sum())
total_test_predictions = len(y_test)
accuracy_percent = accuracy * 100

print(f"Akurasi model: {accuracy_percent:.2f}%")
print(
    f"Artinya, {correct_predictions:,} dari {total_test_predictions:,} post data test "
    f"diprediksi sesuai label {LABEL_SOURCE}."
)


### Penjelasan Cell Kode 17: Akurasi dalam Persen dan Jumlah

Cell ini mengubah nilai akurasi menjadi bahasa yang lebih mudah dibaca. Bukan hanya menampilkan angka desimal, tetapi juga menyebutkan berapa post di data test yang berhasil ditebak sesuai label. Misalnya, akurasi 80% berarti 8 dari 10 data test diprediksi sama dengan label acuannya.


In [ ]:
df_results = df_model.copy()
df_results["model_sentiment"] = best_model.predict(df_results["clean_text"])
df_results["model_sentiment_indonesia"] = df_results["model_sentiment"].map(INDONESIAN_LABELS)

model_counts = df_results["model_sentiment"].value_counts().reindex(SENTIMENT_ORDER, fill_value=0)
model_percent = (model_counts / model_counts.sum() * 100).round(2)

model_result_summary = pd.DataFrame({
    "sentiment": model_counts.index,
    "sentimen_indonesia": [INDONESIAN_LABELS[label] for label in model_counts.index],
    "jumlah_post": model_counts.values,
    "persentase": model_percent.values,
})

display(model_result_summary)


### Penjelasan Cell Kode 18: Jumlah dan Persentase Sentimen dari Model

Cell ini memakai model yang sudah dilatih untuk memberi prediksi sentimen pada seluruh post yang masuk proses modeling. Hasilnya diringkas menjadi tabel: berapa post positif, netral, dan negatif, serta persentasenya dari total post. Inilah tabel utama untuk membaca komposisi opini publik versi model.


In [ ]:
display_columns = [
    col
    for col in [
        "id",
        "created_at_date",
        "author_username",
        "text",
        "clean_text",
        "sentiment",
        "model_sentiment",
        "model_sentiment_indonesia",
        "like_count",
        "retweet_count",
        "reply_count",
        "quote_count",
        "impression_count",
    ]
    if col in df_results.columns
]

posts_by_sentiment = {
    sentiment: df_results[df_results["model_sentiment"].eq(sentiment)][display_columns].copy()
    for sentiment in SENTIMENT_ORDER
}

POSTS_PER_SENTIMENT_DISPLAY = 20

for sentiment in SENTIMENT_ORDER:
    sentiment_name = INDONESIAN_LABELS[sentiment]
    subset = posts_by_sentiment[sentiment]
    percent_value = float(model_percent.loc[sentiment]) if model_counts.sum() else 0.0
    print(f"\nSentimen {sentiment_name}: {len(subset):,} post ({percent_value:.2f}%)")
    display(subset.head(POSTS_PER_SENTIMENT_DISPLAY))


### Penjelasan Cell Kode 19: Data Post untuk Setiap Sentimen

Cell ini memecah hasil prediksi model menjadi tiga kelompok: positif, netral, dan negatif. Untuk tiap kelompok, notebook menampilkan jumlah, persentase, dan contoh data post. Variabel `posts_by_sentiment` menyimpan data lengkapnya, sehingga kita bisa mengambil semua post positif, semua post netral, atau semua post negatif kapan saja.


In [ ]:
OUTPUT_DIR = Path("sentiment_mbg_posts_output")
OUTPUT_DIR.mkdir(exist_ok=True)

exported_files = []
for sentiment, subset in posts_by_sentiment.items():
    sentiment_name = INDONESIAN_LABELS[sentiment]
    output_path = OUTPUT_DIR / f"posts_{sentiment_name}.csv"
    subset.to_csv(output_path, index=False, encoding="utf-8-sig")
    exported_files.append({
        "sentimen": sentiment_name,
        "jumlah_post": len(subset),
        "file": str(output_path),
    })

display(pd.DataFrame(exported_files))


### Penjelasan Cell Kode 20: Menyimpan Data Post per Sentimen

Cell ini menyimpan data lengkap setiap sentimen ke folder `sentiment_mbg_posts_output`. Hasilnya ada tiga file CSV: `posts_positif.csv`, `posts_netral.csv`, dan `posts_negatif.csv`. File ini berguna jika data ingin dibuka di Excel, Google Sheets, atau dipakai untuk laporan.


## 11. Analisis Kata Penting per Kelas

Bagian ini membantu memahami kata atau frasa apa yang paling memengaruhi keputusan model. Untuk orang awam, ini mirip melihat “kata-kata pemicu” yang membuat model condong memilih sentimen tertentu.


### Penjelasan Cell Kode 21: Kata yang Paling Berpengaruh

Cell ini melihat kata atau frasa dengan bobot paling kuat pada model. Hasilnya membantu menjelaskan kenapa model cenderung memilih label tertentu, misalnya kata yang sering mengarah ke sentimen negatif atau positif.


In [ ]:
tfidf = best_model.named_steps["tfidf"]
svm = best_model.named_steps["svm"]
feature_names = np.array(tfidf.get_feature_names_out())

if len(svm.classes_) == 2 and svm.coef_.shape[0] == 1:
    negative_terms = feature_names[np.argsort(svm.coef_[0])[:15]]
    positive_terms = feature_names[np.argsort(svm.coef_[0])[-15:][::-1]]
    print(f"Top terms for {svm.classes_[0]}:")
    print(", ".join(negative_terms))
    print(f"\nTop terms for {svm.classes_[1]}:")
    print(", ".join(positive_terms))
else:
    for class_index, class_name in enumerate(svm.classes_):
        top_indices = np.argsort(svm.coef_[class_index])[-15:][::-1]
        top_terms = feature_names[top_indices]
        print(f"\nTop terms for {class_name}:")
        print(", ".join(top_terms))


## 12. Prediksi Sentimen Teks Baru

Bagian ini menyediakan fungsi sederhana untuk mencoba model pada kalimat baru. Kalimat akan dibersihkan dengan cara yang sama seperti data training, lalu model memberi prediksi sentimen.


### Penjelasan Cell Kode 22: Mencoba Prediksi pada Kalimat Baru

Cell ini membuat fungsi `predict_sentiment` agar kita bisa memasukkan kalimat baru dan melihat prediksi model. Tiga contoh kalimat disediakan untuk menguji apakah model memberi label yang sesuai dengan intuisi.


In [ ]:
def predict_sentiment(texts):
    if isinstance(texts, str):
        texts = [texts]

    cleaned_texts = [clean_x_text(text) for text in texts]
    predictions = best_model.predict(cleaned_texts)

    return pd.DataFrame({
        "text": texts,
        "clean_text": cleaned_texts,
        "predicted_sentiment": predictions,
        "predicted_sentiment_indonesia": [INDONESIAN_LABELS[label] for label in predictions],
    })


sample_texts = [
    "Program MBG sangat membantu anak-anak mendapatkan makanan sehat di sekolah.",
    "Distribusi makan bergizi gratis hari ini berjalan seperti biasa.",
    "Menu MBG basi dan membuat beberapa siswa keracunan, harus dievaluasi serius.",
]

predict_sentiment(sample_texts)


## 13. Simpan Model

Setelah model jadi, pipeline lengkapnya disimpan ke file `.joblib`. Dengan begitu model bisa dipakai lagi di lain waktu tanpa perlu melatih ulang dari awal.


### Penjelasan Cell Kode 23: Menyimpan Model

Cell ini menyimpan model yang sudah dilatih ke file `svm_mbg_sentiment_model.joblib`. File ini berisi pipeline lengkap, daftar label, sumber label, dan catatan konfigurasi agar model bisa dipakai ulang.


In [ ]:
MODEL_PATH = Path("svm_mbg_sentiment_model.joblib")
joblib.dump({
    "model": best_model,
    "labels": list(best_model.named_steps["svm"].classes_),
    "label_source": LABEL_SOURCE,
    "data_path": str(DATA_PATH),
    "dataset_name": DATASET_NAME,
    "topic_name": TOPIC_NAME,
    "use_indonesian_only": USE_INDONESIAN_ONLY,
    "exclude_retweets": EXCLUDE_RETWEETS,
    "cleaning_function_note": "Apply clean_x_text before calling model.predict if not using this notebook helper.",
}, MODEL_PATH)

print(f"Model saved to {MODEL_PATH.resolve()}")


## Kesimpulan

Notebook ini memuat alur lengkap analisis sentimen MBG: membaca data, membersihkan teks, membuat atau memakai label, melatih SVM, mengevaluasi model, lalu menampilkan hasil prediksi dalam bentuk jumlah, persentase, dan data post per sentimen.

Jika dataset belum punya label manual, hasilnya harus dibaca sebagai baseline awal berbasis weak label. Untuk penelitian atau laporan final yang lebih kuat, sebaiknya tambahkan kolom `sentiment` hasil anotasi manusia, lalu jalankan ulang notebook agar model belajar dari label yang lebih valid.
